In [ ]:
import requests
import pandas as pd

URL = "https://www.smartgriddashboard.com/DashboardService.svc/data"
HEADERS = {"User-Agent": "Mozilla/5.0 (research project; wind curtailment analysis)"}

areas = ["windactual", "windforecast", "demandactual",
         "generationactual", "SnspALL", "interconnection"]

for area in areas:
    params = {
        "area": area,
        "region": "ALL",
        "datefrom": "13-Feb-2024 00:00",
        "dateto": "13-Feb-2024 23:59",
    }
    r = requests.get(URL, params=params, headers=HEADERS, timeout=30)
    print(f"\n=== {area} | status {r.status_code} ===")
    try:
        rows = pd.DataFrame(r.json()["Rows"])
        print("rows:", len(rows))
        print(rows.head(3))
    except Exception as e:
        print("could not read rows:", e)
        

In [ ]:
r = requests.get(URL, params={
    "area": "interconnection",
    "region": "ALL",
    "datefrom": "13-Feb-2024 00:00",
    "dateto": "13-Feb-2024 23:59",
}, headers=HEADERS, timeout=30)

ic = pd.DataFrame(r.json()["Rows"])

print(ic.groupby(["FieldName", "Region"]).agg(
    rows=("Value", "size"),
    non_null=("Value", "count"),
))
print(ic.dtypes)

In [ ]:
dates = ["13-Feb-2023", "13-Aug-2024", "13-Feb-2025",
         "13-Aug-2025", "13-Feb-2026", "01-Oct-2026"]

for d in dates:
    r = requests.get(URL, params={
        "area": "interconnection",
        "region": "ALL",
        "datefrom": f"{d} 00:00",
        "dateto": f"{d} 23:59",
    }, headers=HEADERS, timeout=30)
    ic = pd.DataFrame(r.json()["Rows"])
    print(f"\n=== {d} | status {r.status_code} | rows {len(ic)} ===")
    if len(ic):
        print(ic.groupby("FieldName")["Value"].count())

In [ ]:
import time

dates = ["13-Feb-2025", "13-Aug-2025", "13-Feb-2026", "01-Oct-2026"]

for d in dates:
    r = requests.get(URL, params={
        "area": "interconnection",
        "region": "ALL",
        "datefrom": f"{d} 00:00",
        "dateto": f"{d} 23:59",
    }, headers=HEADERS, timeout=30)

    print(f"\n=== {d} | status {r.status_code} | length {len(r.text)} ===")
    print("Start of reply:", r.text[:150])

    try:
        ic = pd.DataFrame(r.json()["Rows"])
        print(ic.groupby("FieldName")["Value"].count())
    except Exception as e:
        print("could not read JSON:", e)

    time.sleep(2)

In [ ]:
import time

areas = ["windactual", "windforecast", "demandactual", "generationactual", "SnspALL"]
dates = ["13-Feb-2021", "13-Feb-2022", "13-Feb-2023",
         "13-Feb-2024", "13-Feb-2025", "13-Feb-2026"]

results = {}

for d in dates:
    results[d] = {}
    for area in areas:
        r = requests.get(URL, params={
            "area": area,
            "region": "ALL",
            "datefrom": f"{d} 00:00",
            "dateto": f"{d} 23:59",
        }, headers=HEADERS, timeout=30)
        try:
            rows = pd.DataFrame(r.json()["Rows"])
            results[d][area] = int(rows["Value"].count()) if len(rows) else 0
        except Exception:
            results[d][area] = f"ERR {r.status_code}"
        time.sleep(1)

print(pd.DataFrame(results).T)

In [ ]:
def fetch_count(area, d, tries=4):
    for attempt in range(tries):
        r = requests.get(URL, params={
            "area": area,
            "region": "ALL",
            "datefrom": f"{d} 00:00",
            "dateto": f"{d} 23:59",
        }, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            try:
                rows = pd.DataFrame(r.json()["Rows"])
                return int(rows["Value"].count()) if len(rows) else 0
            except Exception:
                pass
        time.sleep(2 * (attempt + 1))
    return f"ERR {r.status_code}"

for d in results:
    for area in results[d]:
        if isinstance(results[d][area], str):
            results[d][area] = fetch_count(area, d)
            time.sleep(1)

print(pd.DataFrame(results).T)

In [ ]:
months = ["01-Mar-2021", "01-Apr-2021", "01-May-2021", "01-Jun-2021",
          "01-Jul-2021", "01-Aug-2021", "01-Sep-2021", "01-Oct-2021",
          "01-Nov-2021", "01-Dec-2021", "01-Jan-2022"]

for d in months:
    print(d, fetch_count("SnspALL", d))
    time.sleep(1)

In [ ]:
for day in range(14, 29):
    d = f"{day:02d}-Feb-2021"
    print(d, fetch_count("SnspALL", d))
    time.sleep(1)

In [ ]:
for d in ["30-Mar-2024", "31-Mar-2024", "26-Oct-2024", "27-Oct-2024"]:
    print(d, "wind:", fetch_count("windactual", d), "| SNSP:", fetch_count("SnspALL", d))
    time.sleep(1)

In [ ]:
r = requests.get(URL, params={
    "area": "windactual",
    "region": "ALL",
    "datefrom": "27-Oct-2024 00:00",
    "dateto": "27-Oct-2024 23:59",
}, headers=HEADERS, timeout=30)

w = pd.DataFrame(r.json()["Rows"])
print(w.iloc[0:14][["EffectiveTime", "Value"]])
print("unique times:", w["EffectiveTime"].nunique(), "of", len(w))
print("last row:", w["EffectiveTime"].iloc[-1])

In [ ]:
def fetch_rows(area, date_from, date_to, tries=4):
    for attempt in range(tries):
        r = requests.get(URL, params={
            "area": area,
            "region": "ALL",
            "datefrom": date_from,
            "dateto": date_to,
        }, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            try:
                return pd.DataFrame(r.json()["Rows"])
            except Exception:
                pass
        print(f"attempt {attempt + 1} failed, status {r.status_code}")
        time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"gave up after {tries} tries, last status {r.status_code}")

w = fetch_rows("windactual", "27-Oct-2024 00:00", "27-Oct-2024 23:59")

print(w.iloc[0:14][["EffectiveTime", "Value"]])
print("unique times:", w["EffectiveTime"].nunique(), "of", len(w))
print("last row:", w["EffectiveTime"].iloc[-1])

In [ ]:
tests = [("13-Feb-2024", "windactual"),
         ("26-Oct-2024", "windactual"),
         ("27-Oct-2024", "windactual")]

for d, area in tests:
    r = requests.get(URL, params={
        "area": area,
        "region": "ALL",
        "datefrom": f"{d} 00:00",
        "dateto": f"{d} 23:59",
    }, headers=HEADERS, timeout=30)
    print(d, area, "| status", r.status_code, "| reply starts:", r.text[:120])
    time.sleep(3)

In [ ]:
w = fetch_rows("windactual", "27-Oct-2024 00:00", "27-Oct-2024 23:59")

print(w.iloc[0:14][["EffectiveTime", "Value"]])
print("unique times:", w["EffectiveTime"].nunique(), "of", len(w))
print("last row:", w["EffectiveTime"].iloc[-1])

In [ ]:
n = fetch_rows("windactual", "27-Oct-2024 00:45", "27-Oct-2024 02:15")
print(n[["EffectiveTime", "Value"]])
print("rows:", len(n))